In [4]:
import pytest
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import uuid

# Jupyter 노트북용 경로 설정
current_dir = Path(os.getcwd())  # 현재 작업 디렉토리 (mas 폴더)

# team2-fastapi 폴더를 sys.path에 추가 (scentpick의 부모)
team2_fastapi_root = current_dir.parent.parent  # mas -> scentpick -> team2-fastapi
sys.path.insert(0, str(team2_fastapi_root))

print(f"📁 현재 디렉토리: {current_dir}")
print(f"📁 프로젝트 루트: {team2_fastapi_root}")
print(f"📁 sys.path에 추가됨: {team2_fastapi_root}\n")

# 이제 import 가능
from scentpick.mas.perfume_chatbot import app

# 테스트 결과 저장 경로
RESULTS_DIR = current_dir / "test_results"
RESULTS_DIR.mkdir(exist_ok=True)

class TestSupervisorRouting:
    """수퍼바이저 라우팅 테스트"""
    
    def __init__(self):
        """초기화"""
        self.results = []
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    def invoke_and_record(self, query: str, rec_context: str = "(none)", 
                         last_agent: str = None, test_name: str = ""):
        """그래프 실행 및 결과 기록"""
        initial_state = {
            "messages": [{"role": "user", "content": query}],
            "user_query": query,
            "rec_context": rec_context,
            "last_agent": last_agent,
            "image_url": None
        }
        
        # LangGraph config에 thread_id 추가 (체크포인터 사용 시 필수)
        config = {
            "configurable": {
                "thread_id": str(uuid.uuid4())  # 각 테스트마다 고유 ID
            }
        }
        
        try:
            # LangGraph invoke with config
            result = app.invoke(initial_state, config=config)
            
            # 마지막 메시지에서 응답 추출
            last_message = result.get("messages", [])[-1] if result.get("messages") else {}
            response_content = last_message.get("content", "") if isinstance(last_message, dict) else str(last_message)
            
            record = {
                "test_name": test_name,
                "query": query,
                "rec_context": rec_context,
                "last_agent": last_agent,
                "routed_to": result.get("next_agent", result.get("last_agent", "unknown")),
                "intent": result.get("intent", "unknown"),
                "supervisor_output": result.get("supervisor_output", {}),
                "response": response_content[:500],
                "success": True
            }
        except Exception as e:
            import traceback
            record = {
                "test_name": test_name,
                "query": query,
                "rec_context": rec_context,
                "last_agent": last_agent,
                "error": str(e),
                "traceback": traceback.format_exc(),
                "success": False
            }
        
        self.results.append(record)
        return record
    
    def save_results(self, node_name: str):
        """결과를 텍스트 파일로 저장"""
        filename = RESULTS_DIR / f"{node_name}_{self.timestamp}.txt"
        
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"{'='*80}\n")
            f.write(f"테스트 노드: {node_name}\n")
            f.write(f"실행 시간: {self.timestamp}\n")
            f.write(f"총 테스트: {len(self.results)}개\n")
            f.write(f"{'='*80}\n\n")
            
            success_count = sum(1 for r in self.results if r.get("success", False))
            f.write(f"✅ 성공: {success_count}/{len(self.results)}\n")
            f.write(f"❌ 실패: {len(self.results) - success_count}/{len(self.results)}\n\n")
            
            for i, result in enumerate(self.results, 1):
                f.write(f"\n{'─'*80}\n")
                f.write(f"테스트 #{i}: {result['test_name']}\n")
                f.write(f"{'─'*80}\n")
                f.write(f"📝 쿼리: {result['query']}\n")
                f.write(f"📋 REC_CONTEXT: {result['rec_context']}\n")
                f.write(f"👤 LAST_AGENT: {result['last_agent']}\n\n")
                
                if result.get("success"):
                    f.write(f"🎯 라우팅됨: {result['routed_to']}\n")
                    f.write(f"🔍 의도(Intent): {result['intent']}\n")
                    
                    if result.get("supervisor_output"):
                        f.write(f"\n📊 Supervisor Output:\n")
                        supervisor = result["supervisor_output"]
                        if isinstance(supervisor, dict):
                            f.write(f"  - next: {supervisor.get('next', 'N/A')}\n")
                            f.write(f"  - intent: {supervisor.get('intent', 'N/A')}\n")
                            f.write(f"  - followup: {supervisor.get('followup', False)}\n")
                            f.write(f"  - confidence: {supervisor.get('confidence', 0.0)}\n")
                            f.write(f"  - facet_count: {supervisor.get('facet_count', 0)}\n")
                            f.write(f"  - reason: {supervisor.get('reason', 'N/A')}\n")
                            
                            if supervisor.get('facets'):
                                f.write(f"  - facets: {json.dumps(supervisor['facets'], ensure_ascii=False, indent=4)}\n")
                            if supervisor.get('scent_vibe'):
                                f.write(f"  - scent_vibe: {supervisor.get('scent_vibe')}\n")
                    
                    f.write(f"\n💬 응답 미리보기:\n{result['response']}\n")
                else:
                    f.write(f"❌ 에러 발생:\n{result.get('error', 'Unknown error')}\n")
                    if result.get('traceback'):
                        f.write(f"\n스택 트레이스:\n{result['traceback']}\n")
                
                f.write("\n")
        
        print(f"✅ 결과 저장됨: {filename}")


    # ============================================================================
    # LLM_parser 테스트 (10개)
    # ============================================================================
    def test_llm_parser_queries(self):
        """LLM_parser 노드 테스트"""
        self.results = []
        queries = [
            ("샤넬 향수 추천해줘", "Brand facet"),
            ("여름 향수 추천해줘", "Season facet"),
            ("여성용 향수 추천해줘", "Gender facet"),
            ("50ml 향수 추천해줘", "Size facet"),
            ("밤에 쓰는 향수 추천해줘", "Day/Night facet"),
            ("오드 뚜왈렛 추천해줘", "Concentration facet"),
            ("디올 남성용 100ml 향수 추천", "Multiple facets"),
            ("입생로랑 여성용 50ml 겨울용 향수 추천해줘. 가격도 알려줘", "Multi-facet + price"),
            ("조말론 데이타임용 향수 추천", "Brand + day_night"),
            ("톰포드 오드 퍼퓸 추천해줘", "Brand + concentration"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, test_name=f"LLM_parser - {test_name}")
        
        self.save_results("LLM_parser")


    # ============================================================================
    # FAQ_agent 테스트 (10개)
    # ============================================================================
    def test_faq_agent_queries(self):
        """FAQ_agent 노드 테스트"""
        self.results = []
        queries = [
            ("오드 뚜왈렛 뜻이 뭐야?", "Definition query"),
            ("오드퍼퓸과 오드뚜왈렛의 차이는?", "Difference query"),
            ("향수 지속력은 어떻게 결정돼?", "Knowledge query"),
            ("탑노트가 뭐야?", "Term definition"),
            ("향수는 어디에 뿌리는 게 좋아?", "Usage tip"),
            ("향수 보관법 알려줘", "Storage info"),
            ("니치향수가 뭐야?", "Category definition"),
            ("향수 농도 종류 알려줘", "Classification"),
            ("향수는 왜 시간이 지나면 향이 변해?", "Concept explanation"),
            ("향수와 퍼퓸의 차이는?", "Terminology"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, test_name=f"FAQ_agent - {test_name}")
        
        self.save_results("FAQ_agent")


    # ============================================================================
    # human_fallback 테스트 (10개)
    # ============================================================================
    def test_human_fallback_queries(self):
        """human_fallback 노드 테스트"""
        self.results = []
        queries = [
            ("데오드란트 추천해줘", "Non-perfume: deodorant"),
            ("틴트 추천해줘", "Non-perfume: tint"),
            ("섬유유연제 추천해줘", "Non-perfume: fabric softener"),
            ("디퓨저 추천해줘", "Non-perfume: diffuser"),
            ("오늘 날씨 어때?", "Off-topic: weather"),
            ("피자 레시피 알려줘", "Off-topic: food"),
            ("바디미스트 추천해줘", "Non-perfume: body mist"),
            ("룸스프레이 추천해줘", "Non-perfume: room spray"),
            ("파이썬 코딩 도와줘", "Off-topic: programming"),
            ("샴푸 추천해줘", "Non-perfume: shampoo"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, test_name=f"human_fallback - {test_name}")
        
        self.save_results("human_fallback")


    # ============================================================================
    # price_agent 테스트 (10개)
    # ============================================================================
    def test_price_agent_queries(self):
        """price_agent 노드 테스트"""
        self.results = []
        queries = [
            ("10만원대 향수 추천해줘", "Price-only: 만원대"),
            ("5만원 이하 향수 추천", "Price-only: budget limit"),
            ("샤넬 블루 드 샤넬 가격은?", "Specific product price"),
            ("디올 소바쥬 얼마야?", "Price inquiry"),
            ("향수 가격 알려줘", "General price query"),
            ("최저가 향수 추천", "Cheapest query"),
            ("할인하는 향수 있어?", "Discount query"),
            ("15만원에서 20만원 사이 향수", "Price range"),
            ("구매처 알려줘", "Purchase location"),
            ("배송비 포함 얼마야?", "Shipping cost"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, test_name=f"price_agent - {test_name}")
        
        self.save_results("price_agent")


    # ============================================================================
    # ML_agent 테스트 (10개)
    # ============================================================================
    def test_ml_agent_queries(self):
        """ML_agent 노드 테스트"""
        self.results = []
        queries = [
            ("시원한 아쿠아향 향수 추천해줘", "Scent-only: aquatic"),
            ("달달한 향수 추천", "Scent-only: sweet"),
            ("포근한 느낌의 향수", "Scent vibe: cozy"),
            ("상쾌한 시트러스 향수", "Scent-only: citrus"),
            ("우디한 향수 추천", "Scent-only: woody"),
            ("플로럴 무드 향수", "Scent vibe: floral"),
            ("히노키 숲향 나는 향수", "Scent vibe: hinoki forest"),
            ("바닐라 향 좋아해", "Scent preference: vanilla"),
            ("스파이시한 향수 추천", "Scent-only: spicy"),
            ("은은한 머스크향", "Scent vibe: subtle musk"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, test_name=f"ML_agent - {test_name}")
        
        self.save_results("ML_agent")


    # ============================================================================
    # memory_echo 테스트 (10개)
    # ============================================================================
    def test_memory_echo_queries(self):
        """memory_echo 노드 테스트"""
        self.results = []
        queries = [
            ("내가 방금 뭐라 했지?", "Memory recall: what did I say"),
            ("내 마지막 질문 뭐였지?", "Memory recall: last question"),
            ("아까 내가 뭐 물어봤어?", "Memory recall: previous query"),
            ("방금 전에 내가 한 말 뭐야?", "Memory recall: recent utterance"),
            ("내가 처음에 뭐라고 했지?", "Memory recall: first query"),
            ("이전 질문 다시 보여줘", "Memory recall: show previous"),
            ("내가 말한 거 기억해?", "Memory check"),
            ("방금 내 질문 뭐였어?", "Memory recall: just now"),
            ("아까 물어본 거 뭐였지?", "Memory recall: earlier"),
            ("내가 뭐 요청했더라?", "Memory recall: request"),
        ]
        
        rec_context = """1. Chanel Chance Eau Tendre
2. YSL Libre"""
        
        for query, test_name in queries:
            self.invoke_and_record(query, rec_context=rec_context, 
                                 last_agent="FAQ_agent",
                                 test_name=f"memory_echo - {test_name}")
        
        self.save_results("memory_echo")


    # ============================================================================
    # rec_echo 테스트 (10개)
    # ============================================================================
    def test_rec_echo_queries(self):
        """rec_echo 노드 테스트"""
        self.results = []
        queries = [
            ("방금 추천해준 향수 이름이 뭐지?", "Re-show: names"),
            ("아까 추천한 거 다시 보여줘", "Re-show: list"),
            ("추천 목록 다시 알려줘", "Re-show: recap"),
            ("방금 내용 요약해줘", "Re-show: summary"),
            ("그 향수들 이름 뭐였어?", "Re-show: product names"),
            ("추천한 거 뭐였지?", "Re-show: what recommended"),
            ("리스트 다시 보여줘", "Re-show: show list again"),
            ("방금 말한 향수 다시", "Re-show: repeat"),
            ("추천 결과 다시 한번", "Re-show: results again"),
            ("아까 그 향수들 뭐였어?", "Re-show: those perfumes"),
        ]
        
        rec_context = """1. Chanel Bleu de Chanel
2. Dior Sauvage
3. Tom Ford Noir"""
        
        for query, test_name in queries:
            self.invoke_and_record(query, rec_context=rec_context,
                                 last_agent="ML_agent",
                                 test_name=f"rec_echo - {test_name}")
        
        self.save_results("rec_echo")


    # ============================================================================
    # review_agent 테스트 (10개)
    # ============================================================================
    def test_review_agent_queries(self):
        """review_agent 노드 테스트"""
        self.results = []
        queries = [
            ("히노키숲향 향수 추천해주고 가격도 알려줘", "Scent + price"),
            ("달달한 향수 추천, 가격대도 알려줘", "Sweet scent + price"),
            ("시원한 향수 10만원대로", "Fresh scent + price range"),
            ("플로럴 향 좋아하는데 저렴한 거", "Floral + cheap"),
            ("우디향 나는 거 가격 알려줘", "Woody + price"),
            ("포근한 느낌 향수 얼마야?", "Cozy vibe + price"),
            ("바닐라향 향수 구매하고 싶어", "Vanilla + purchase intent"),
            ("머스크향 저렴한 거 추천", "Musk + budget"),
            ("시트러스 향수 가격대 궁금해", "Citrus + price inquiry"),
            ("스파이시한 향수 할인하는 거 있어?", "Spicy + discount"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, test_name=f"review_agent - {test_name}")
        
        self.save_results("review_agent")


    # ============================================================================
    # 팔로업 시나리오 테스트 (10개)
    # ============================================================================
    def test_followup_scenarios(self):
        """팔로업 시나리오 테스트"""
        self.results = []
        rec_context = """1. Chanel Bleu de Chanel
2. Dior Sauvage
3. Tom Ford Noir"""
        
        queries = [
            ("두번째 가격은?", "Price followup #2"),
            ("3번 노트 알려줘", "Detail followup #3"),
            ("첫번째 상세정보", "Detail followup #1"),
            ("2번이랑 3번 비교해줘", "Compare followup"),
            ("그거 어디서 사?", "Purchase followup"),
            ("첫번째꺼 지속력은?", "Longevity followup"),
            ("3번 가격 얼마야?", "Price followup #3"),
            ("두번째 향 어때?", "Scent followup #2"),
            ("1번 할인하는 곳 있어?", "Discount followup #1"),
            ("세번째 리뷰 보여줘", "Review followup #3"),
        ]
        
        for query, test_name in queries:
            self.invoke_and_record(query, rec_context=rec_context,
                                 last_agent="ML_agent",
                                 test_name=f"Followup - {test_name}")
        
        self.save_results("followup_scenarios")


# 사용 예시
print("🚀 테스트 시작...")
tester = TestSupervisorRouting()

# 모든 테스트 실행
print("\n1️⃣ LLM_parser 테스트 시작...")
tester.test_llm_parser_queries()

print("\n2️⃣ FAQ_agent 테스트 시작...")
tester.test_faq_agent_queries()

print("\n3️⃣ human_fallback 테스트 시작...")
tester.test_human_fallback_queries()

print("\n4️⃣ price_agent 테스트 시작...")
tester.test_price_agent_queries()

print("\n5️⃣ ML_agent 테스트 시작...")
tester.test_ml_agent_queries()

print("\n6️⃣ memory_echo 테스트 시작...")
tester.test_memory_echo_queries()

print("\n7️⃣ rec_echo 테스트 시작...")
tester.test_rec_echo_queries()

print("\n8️⃣ review_agent 테스트 시작...")
tester.test_review_agent_queries()

print("\n9️⃣ followup 시나리오 테스트 시작...")
tester.test_followup_scenarios()

print("\n✅ 모든 테스트 완료!")

📁 현재 디렉토리: c:\team2-fastapi\scentpick\mas
📁 프로젝트 루트: c:\team2-fastapi
📁 sys.path에 추가됨: c:\team2-fastapi

🚀 테스트 시작...

1️⃣ LLM_parser 테스트 시작...
🔍 LLM_parser 실행: 샤넬 향수 추천해줘
🔧 run_llm_parser 호출
🔧 run_llm_parser 결과
{"brand": "샤넬", "gender": null, "sizes": null, "season_score": null, "concentration": null, "day_night_score": null, "recommendation_count": null}
🔧 apply_meta_filters 호출
🔧 apply_meta_filters 결과
{"brand": "샤넬", "concentration": null, "day_night_score": null, "gender": null, "season_score": null, "sizes": null}
쿼리 벡터화
pinecone 검색
Pinecone 검색 결과 (메타필터링 컬럼만)
1. brand=샤넬, name=코코 오 드 퍼퓸 리필, gender=Female, size=['60']ml, season=fall, day_night=day, concentration=오 드 퍼퓸
2. brand=샤넬, name=코코 오 드 퍼퓸, gender=Female, size=['35', '50']ml, season=fall, day_night=day, concentration=오 드 퍼퓸
3. brand=샤넬, name=코코 오 드 뚜왈렛, gender=Female, size=['50', '100']ml, season=fall, day_night=day, concentration=오 드 뚜왈렛
추천 후보 (정제된 candidates):
1. 샤넬 - 코코 오 드 퍼퓸 리필 (Noneml) 
2. 샤넬 - 코코 오 드 퍼퓸 (Noneml) 
3. 샤넬 

c:\Users\Playdata2\miniconda3\envs\final-env\Lib\pickle.py:1760: UserWarning: [11:56:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\data\../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)
`torch_dtype` is deprecated! Use `dtype` instead!


🔍 ML_parser 실행: 달달한 향수 추천
🔍 ML_parser 실행: 포근한 느낌의 향수
🔍 ML_parser 실행: 상쾌한 시트러스 향수
🔍 ML_parser 실행: 우디한 향수 추천
🔍 ML_parser 실행: 플로럴 무드 향수
🔍 ML_parser 실행: 히노키 숲향 나는 향수
🔍 ML_parser 실행: 바닐라 향 좋아해
🔍 ML_parser 실행: 스파이시한 향수 추천
🔍 ML_parser 실행: 은은한 머스크향
✅ 결과 저장됨: c:\team2-fastapi\scentpick\mas\test_results\ML_agent_20250929_115030.txt

6️⃣ memory_echo 테스트 시작...
✅ 결과 저장됨: c:\team2-fastapi\scentpick\mas\test_results\memory_echo_20250929_115030.txt

7️⃣ rec_echo 테스트 시작...
🔍 human_fallback 실행: 아까 그 향수들 뭐였어?
✅ 결과 저장됨: c:\team2-fastapi\scentpick\mas\test_results\rec_echo_20250929_115030.txt

8️⃣ review_agent 테스트 시작...


[analyze_rag_results] Error: Expecting value: line 1 column 1 (char 0)
Traceback (most recent call last):
  File "c:\team2-fastapi\scentpick\mas\nodes\review_agent_node.py", line 212, in analyze_rag_results
    parsed = json.loads(getattr(response, "content", "{}"))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Playdata2\miniconda3\envs\final-env\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Playdata2\miniconda3\envs\final-env\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Playdata2\miniconda3\envs\final-env\Lib\json\decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)


🔍 ML_parser 실행: 바닐라향 향수 구매하고 싶어


[analyze_rag_results] Error: Expecting value: line 1 column 1 (char 0)
Traceback (most recent call last):
  File "c:\team2-fastapi\scentpick\mas\nodes\review_agent_node.py", line 212, in analyze_rag_results
    parsed = json.loads(getattr(response, "content", "{}"))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Playdata2\miniconda3\envs\final-env\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Playdata2\miniconda3\envs\final-env\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Playdata2\miniconda3\envs\final-env\Lib\json\decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)


✅ 결과 저장됨: c:\team2-fastapi\scentpick\mas\test_results\review_agent_20250929_115030.txt

9️⃣ followup 시나리오 테스트 시작...
🔍 human_fallback 실행: 3번 노트 알려줘
🔍 ML_parser 실행: 첫번째 상세정보
🔍 ML_parser 실행: 2번이랑 3번 비교해줘
🔍 human_fallback 실행: 첫번째꺼 지속력은?
🔍 human_fallback 실행: 두번째 향 어때?
🔍 human_fallback 실행: 세번째 리뷰 보여줘
✅ 결과 저장됨: c:\team2-fastapi\scentpick\mas\test_results\followup_scenarios_20250929_115030.txt

✅ 모든 테스트 완료!
